## Get Dataset

In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

In [2]:
%pip install -qqq --upgrade ultralytics pyyaml transformers opencv-python-headless seaborn timm "numpy==1.26.4" "scikit-learn==1.3.2" huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 

## Setup Kaggle

In [3]:
import json, cv2
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from collections import Counter
from transformers import ViTForImageClassification
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from PIL import Image, ImageOps
from datetime import datetime, timedelta
from glob import glob
import os, shutil, yaml, re, unicodedata
from shutil import move
from collections import defaultdict
import matplotlib.pyplot as plt
from torch.amp import autocast
from timm.data.mixup import Mixup
from tqdm.auto import tqdm
from tqdm.notebook import tqdm
#  INPUT datasets
DATASETS = [
    "/kaggle/input/bracol-for-yolov8-detection/BRACOL_REVIEWED_ANNOTATIONS/BRACOL_REVIEWED",
    "/kaggle/input/coffee-leaves/BRACOL-VALIDADO.v11i.yolov11",
    "/kaggle/input/coffee-leaves/Coffee Leaf Diseases - OD.v10i.yolov11",
    "/kaggle/input/coffee-leaves/Coffee Leaf.v7-train-valid-test-70-20-10.yolov11",
    "/kaggle/input/coffee-leaves/Coffee leaf diseases classification.v5i.yolov11",
    "/kaggle/input/coffee-leaves/Coffee_Disease_BRACOL.v3i.yolov11",
    "/kaggle/input/coffee-leaves/Folhas.v8i.yolov11",
    "/kaggle/input/coffee-leaves/Leaf plants clasification.v13i.yolov11",
    "/kaggle/input/coffee-leaves/My First Project.v23i.yolov11",
    "/kaggle/input/coffee-leaves/My First Project.v2i.yolov11",
    "/kaggle/input/coffee-leaves/NeuralCoffe.v15i.yolov11",
    "/kaggle/input/coffee-leaves/cashew",
    "/kaggle/input/coffee-leaves/coffee leaf disease.v3i.yolov11",
    "/kaggle/input/coffee-leaves/coffee leaf.v7i.yolov11",
    "/kaggle/input/coffee-leaves/coffee leaves.v1i.yolov11",
    "/kaggle/input/coffee-leaves/coffee.v2i.yolov11",
    "/kaggle/input/coffee-leaves/datasets",
    "/kaggle/input/coffee-leaves/nhan_dien_benh_tren_la_ca_phe.v2i.yolov11"
]

#  Canonical classes
CANONICAL_NAMES = [
    "nam_ri_sat", 
    "sau_duc_la", 
    "khoe_manh", 
    "phoma", 
    "than_thu",
]
SYNONYMS = {
    "nam_ri_sat": {"rust", "leaf rust", "leaf_rust", "Rust"}, #"nam ri sat", "nam_ri_sat", "coffee_rust"
    "sau_duc_la": {"miner", "Miner", "leaf_miner", "leafminor", "leaf minor", "leaf_minor", "Mineiro", "mineiro"}, #"coffee_miner"
    "khoe_manh": {"healthy"}, #"Healthy", "coffee_healthy", "Healthy_Leaf", "healthy_leaf", "Healthy-Leaf"
    "phoma": {"Phoma"}, #"coffee_phoma", "phoma"
    "than_thu": {"anthracnose", "Anthracnose", "Antracnose"}
}

#  Paths 
MERGED_ROOT = "/kaggle/working/merged"
ROI_ROOT    = "/kaggle/working/roi"
CKPT_ROOT   = "/kaggle/working/checkpoints"
#  YOLO train settings 
YOLO_MODEL = "yolo11l.pt"
IMGSZ_YOLO = 640
EPOCHS_YOLO = 35
DEVICE_YOLO = 'cuda'
WORKERS_YOLO = os.cpu_count() // 2
#  ViT settings 
VIT_MODEL_NAME = 'google/vit-base-patch16-224'
ROI_SIZE = 224
EPOCHS_VIT = 30
BASE_LR_VIT = 1e-6
BATCH_VIT = 64
MIXUP = 0.4
LABEL_SMOOTH = 0.05
WEIGHT_DECAY = 0.07
def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))

def norm_name(name: str) -> str:
    n = strip_accents(name.strip())
    n = re.sub(r"[^a-zA-Z0-9\s\-]+", "_", n).strip('_')
    print(n)
    return n

def canonicalize(name: str):
    n = norm_name(name)
    for cano, pool in SYNONYMS.items():
        if n in pool or n == cano:
            return cano
    return None
print(datetime.utcnow() + timedelta(hours=7))

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

2026-03-25 08:35:24.451388


## Cấu hình dataset, hợp nhất các bộ datasets và lọc ROI

In [4]:

for split in ["train", "valid", "test"]:
    Path(f"{MERGED_ROOT}/{split}/images").mkdir(parents=True, exist_ok=True)
    Path(f"{MERGED_ROOT}/{split}/labels").mkdir(parents=True, exist_ok=True)

SPLIT_ALIASES = {
    "train": ["train"],
    "valid":  ["valid", "val"],
    "test": ["test"]
}

def resolve_split_dirs(ds_root: Path):
    #Trả về dict: split_chuan ('train'/'valid'/'test') -> (img_dir, lbl_dir)
    #Hỗ trợ 2 layout:
    #  A) Roboflow:    <root>/<split>/images, <root>/<split>/labels
    #  B) Ultralytics: <root>/images/<split>, <root>/labels/<split>
    
    results = {}
    # --- Layout: <root>/<alias>/{images,labels} ---
    for split_std, aliases in SPLIT_ALIASES.items():
        for alias in aliases:
            img_a = ds_root / alias / "images"
            lbl_a = ds_root / alias / "labels"
            if img_a.exists() and lbl_a.exists():
                results[split_std] = (img_a, lbl_a)
                break
    return results

stats = {s: defaultdict(int) for s in ["train","valid","test"]}

for ds in DATASETS:
    ds_root = Path(ds)
    if not ds_root.exists():
        print(f"[WARN] Dataset không tồn tại: {ds}")
        continue
    yaml_path = ds_root / "data.yaml"
    names = None
    if yaml_path.exists():
        with open(yaml_path,'r') as f:
            meta = yaml.safe_load(f)
        names = meta.get('names')
        if isinstance(names, dict):
            try:
                names = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]
            except Exception:
                # fallback: sắp theo khóa như chuỗi
                names = [names[k] for k in sorted(names.keys())]
    id_map = {}
    if names is not None:
        for old_id, name in enumerate(names):
            cano = canonicalize(str(name))
            if cano is None:
                print(f"[WARN] '{name}' không map -> bỏ bbox lớp này.")
                continue
            id_map[old_id] = CANONICAL_NAMES.index(cano)
    prefix = ds_root.name.replace(' ','_')    
    # Phát hiện cấu trúc split và lặp
    split_dirs = resolve_split_dirs(ds_root)

    for split_std, (img_dir, lbl_dir) in split_dirs.items():
    
        for txt in lbl_dir.glob('*.txt'):
            lines_out = []
    
            with open(txt,'r') as f:
                for ln in f:
                    ln = ln.strip()
                    if not ln:
                        continue
                    parts = ln.split()
                    old_id = int(parts[0])
                    if old_id not in id_map:
                        stats[split_std]['labels_dropped'] += 1
                        continue
                    parts[0] = str(id_map[old_id])
                    lines_out.append(' '.join(parts))
    
            # CHỈ GIỮ ẢNH NẾU CÓ LABEL HỢP LỆ
            if not lines_out:
                continue
    
            img_name = txt.stem + '.jpg'   # fallback
            for ext in ['.jpg','.jpeg','.png','.bmp','.webp']:
                cand = img_dir / (txt.stem + ext)
                if cand.exists():
                    img_name = cand.name
                    break
    
            src_img = img_dir / img_name
            if not src_img.exists():
                continue
    
            # copy image
            shutil.copy2(
                src_img,
                Path(f"{MERGED_ROOT}/{split_std}/images") / f"{prefix}_{img_name}"
            )
    
            # write label
            out_lbl = Path(f"{MERGED_ROOT}/{split_std}/labels") / f"{prefix}_{txt.name}"
            with open(out_lbl,'w') as g:
                g.write('\n'.join(lines_out)+'\n')
    
            stats[split_std]['images'] += 1
            stats[split_std]['labels_kept'] += 1

merged_yaml = {
    'path': ROI_ROOT,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    #'nc': {len(CANONICAL_NAMES)},
    'names': {i:n for i,n in enumerate(CANONICAL_NAMES)}
}
with open(Path("/kaggle/working")/'merged_data.yaml','w',encoding='utf-8') as f:
    yaml.safe_dump(merged_yaml, f, sort_keys=False, allow_unicode=True)
print('MERGE DONE')
for s in ["train","valid","test"]:
    print(s, dict(stats[s]))

print(datetime.utcnow() + timedelta(hours=7))


Cercospora
[WARN] 'Cercospora' không map -> bỏ bbox lớp này.
Miner
Phoma
Rust
Cercospora
[WARN] 'Cercospora' không map -> bỏ bbox lớp này.
Miner
Phoma
Rust
abiotic disorder
[WARN] 'abiotic disorder' không map -> bỏ bbox lớp này.
cercospora
[WARN] 'cercospora' không map -> bỏ bbox lớp này.
healthy
rust
sooty mold
[WARN] 'sooty mold' không map -> bỏ bbox lớp này.
healthy
miner
phoma
rust
cerscospora
[WARN] 'cerscospora' không map -> bỏ bbox lớp này.
healthy
miner
phoma
rust
Cercosporiose
[WARN] 'Cercosporiose' không map -> bỏ bbox lớp này.
Ferrugem
[WARN] 'Ferrugem' không map -> bỏ bbox lớp này.
Mineiro
Phoma
cercosporiose
[WARN] 'cercosporiose' không map -> bỏ bbox lớp này.
healthy
miner
phoma
rust
Algal
[WARN] 'Algal' không map -> bỏ bbox lớp này.
Antracnose
Bacterial-Leaf-Blight
[WARN] 'Bacterial-Leaf-Blight' không map -> bỏ bbox lớp này.
Bacterial-Spot
[WARN] 'Bacterial-Spot' không map -> bỏ bbox lớp này.
Bird-Eye-Spot
[WARN] 'Bird-Eye-Spot' không map -> bỏ bbox lớp này.
Black-Rot
[W

In [5]:

for split in ['train','valid','test']:
    imgs = list(Path(f"{MERGED_ROOT}/{split}/images").iterdir())
    lbls = list(Path(f"{MERGED_ROOT}/{split}/labels").iterdir())
    print(split, "images:", len(imgs), "labels:", len(lbls))
# Tạo cấu trúc ROI_ROOT
for split in ['train','valid','test']:
    Path(f"{ROI_ROOT}/{split}/images").mkdir(parents=True, exist_ok=True)
    Path(f"{ROI_ROOT}/{split}/labels").mkdir(parents=True, exist_ok=True)

roi_stats = {s: defaultdict(int) for s in ['train','valid','test']}

for split in ['train','valid','test']:
    src_images = Path(f"{MERGED_ROOT}/{split}/images")
    src_labels = Path(f"{MERGED_ROOT}/{split}/labels")

    dst_images = Path(f"{ROI_ROOT}/{split}/images")
    dst_labels = Path(f"{ROI_ROOT}/{split}/labels")

    for img_path in src_images.iterdir():
        if not img_path.is_file():
            continue
        if img_path.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp','.webp'}:
            continue

        label_path = src_labels / (img_path.stem + '.txt')
        if not label_path.exists():
            roi_stats[split]['no_label'] += 1
            continue

        # kiểm tra label có class hợp lệ không
        valid = False
        with open(label_path, 'r') as f:
            for ln in f:
                parts = ln.strip().split()
                if len(parts) < 5:
                    continue
                cid = int(parts[0])
                if 0 <= cid < len(CANONICAL_NAMES):
                    valid = True
                    break

        if not valid:
            roi_stats[split]['no_valid_class'] += 1
            continue

        shutil.copy2(img_path, dst_images / img_path.name)
        shutil.copy2(label_path, dst_labels / label_path.name)
        roi_stats[split]['kept'] += 1


train images: 17955 labels: 17955
valid images: 2562 labels: 2562
test images: 3152 labels: 3152


In [6]:
# Đảo ngược mẫu valid <-> test cho các nhãn (trừ 'than_thu')

def swap_valid_test_except_than_thu(roi_root, canonical_names):
    valid_img_dir = Path(roi_root) / 'valid' / 'images'
    valid_lbl_dir = Path(roi_root) / 'valid' / 'labels'
    test_img_dir = Path(roi_root) / 'test' / 'images'
    test_lbl_dir = Path(roi_root) / 'test' / 'labels'
    # Lấy danh sách file nhãn valid/test
    valid_lbls = list(valid_lbl_dir.glob('*.txt'))
    test_lbls = list(test_lbl_dir.glob('*.txt'))
    # Đảo nhãn (trừ than_thu)
    def is_than_thu(lbl_path):
        with open(lbl_path, 'r') as f:
            for ln in f:
                parts = ln.strip().split()
                if len(parts) < 1: continue
                cid = int(parts[0])
                if canonical_names[cid] == 'than_thu':
                    return True
        return False
    # Di chuyển valid -> test nếu không phải than_thu
    for lbl in valid_lbls:
        if not is_than_thu(lbl):
            img_name = lbl.stem + '.jpg'
            img_path = valid_img_dir / img_name
            if img_path.exists():
                move(str(img_path), str(test_img_dir / img_path.name))
            move(str(lbl), str(test_lbl_dir / lbl.name))
    # Di chuyển test -> valid nếu không phải than_thu
    for lbl in test_lbls:
        if not is_than_thu(lbl):
            img_name = lbl.stem + '.jpg'
            img_path = test_img_dir / img_name
            if img_path.exists():
                move(str(img_path), str(valid_img_dir / img_path.name))
            move(str(lbl), str(valid_lbl_dir / lbl.name))
swap_valid_test_except_than_thu(ROI_ROOT, CANONICAL_NAMES)

In [7]:
for s in ['train','valid','test']:
    print(f"[ROI {s}]", dict(roi_stats[s]))

def count_per_class(root):
    counts = {c: 0 for c in CANONICAL_NAMES}

    label_dir = Path(root) / "labels"
    if not label_dir.exists():
        return counts

    for txt in label_dir.glob("*.txt"):
        present = set()
        with open(txt, "r") as f:
            for ln in f:
                parts = ln.strip().split()
                if len(parts) < 5:
                    continue
                cid = int(parts[0])
                if 0 <= cid < len(CANONICAL_NAMES):
                    present.add(CANONICAL_NAMES[cid])

        # mỗi class chỉ +1 / ảnh
        for c in present:
            counts[c] += 1

    return counts

info = 'info.txt'
train_write = count_per_class(f"{ROI_ROOT}/train")
print("ROI train counts:", train_write)
valid_write = count_per_class(f"{ROI_ROOT}/valid")
print("ROI valid counts:", valid_write)
test_write = count_per_class(f"{ROI_ROOT}/test")
print("ROI test counts:", test_write)

print(datetime.utcnow() + timedelta(hours=7))

if os.path.exists(MERGED_ROOT):
    shutil.rmtree(MERGED_ROOT)
    print(f"Đã xóa thư mục{MERGED_ROOT}")

[ROI train] {'kept': 17955}
[ROI valid] {'kept': 2562}
[ROI test] {'kept': 3152}
ROI train counts: {'nam_ri_sat': 3967, 'sau_duc_la': 2377, 'khoe_manh': 2049, 'phoma': 4231, 'than_thu': 5952}
ROI valid counts: {'nam_ri_sat': 933, 'sau_duc_la': 944, 'khoe_manh': 344, 'phoma': 867, 'than_thu': 599}
ROI test counts: {'nam_ri_sat': 704, 'sau_duc_la': 491, 'khoe_manh': 319, 'phoma': 567, 'than_thu': 277}
2026-03-25 08:50:08.830690
Đã xóa thư mục/kaggle/working/merged


In [8]:
with open(Path('/kaggle/working') / info,'w',encoding='utf-8') as f: f.write(json.dumps(train_write, ensure_ascii=False) + '\n')
with open(Path('/kaggle/working') / info,'a',encoding='utf-8') as f: f.write(json.dumps(valid_write, ensure_ascii=False) + '\n')
with open(Path('/kaggle/working') / info,'a',encoding='utf-8') as f: f.write(json.dumps(test_write, ensure_ascii=False) + '\n')